<a href="https://colab.research.google.com/github/Michael-AI-Dam/Flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.sql(f"CREATE OR REPLACE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

base = "hf://datasets/FlyRank/internship-warehouse"
print("Connected — secret registered.")

Connected — secret registered.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one page (content_hash_id), for one client (client_hash_id), on one
report_date, within fact_content_daily_performance. Time window: month=2026-03
(mid-panel, not the sealed final month 2026-06). Confirmed via the query below:
GSC fields (impressions, clicks, position) are populated, but GA4 fields are
entirely NULL for at least this client, since client_has_ga4 = false for them —
this table isn't guaranteed complete across both data sources per client.

In [9]:
con.sql(f"""
    SELECT *
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
    LIMIT 5
""")

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬───────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_clau

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [11]:
# Grain check: one row per (client, content, date)?
con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS row_count
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""")
# 0 rows back = grain confirmed: no duplicate (client, content, date) combos

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬─────────────────┬─────────────┬───────────┐
│ client_hash_id │ content_hash_id │ report_date │ row_count │
│    varchar     │     varchar     │    date     │   int64   │
├────────────────┴─────────────────┴─────────────┴───────────┤
│                           0 rows                           │
└────────────────────────────────────────────────────────────┘

In [12]:
# Row count + date span + distinct pages/clients
con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           MIN(report_date) AS first_date,
           MAX(report_date) AS last_date,
           COUNT(DISTINCT content_hash_id) AS distinct_pages,
           COUNT(DISTINCT client_hash_id) AS distinct_clients
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
""")

┌────────────┬────────────┬────────────┬────────────────┬──────────────────┐
│ total_rows │ first_date │ last_date  │ distinct_pages │ distinct_clients │
│   int64    │    date    │    date    │     int64      │      int64       │
├────────────┼────────────┼────────────┼────────────────┼──────────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │         331437 │               55 │
└────────────┴────────────┴────────────┴────────────────┴──────────────────┘

In [13]:
# Availability: how many rows have usable GSC data vs GA4 data
con.sql(f"""
    SELECT
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        COUNT(*) AS total_rows
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬────────────────────┬────────────┐
│ gsc_available_rows │ ga4_available_rows │ total_rows │
│       int128       │       int128       │   int64    │
├────────────────────┼────────────────────┼────────────┤
│            3611061 │             413966 │    9841378 │
└────────────────────┴────────────────────┴────────────┘

- Grain check: confirmed — 0 duplicate rows for any (client, content, report_date)
  combination. One row genuinely is one page, for one client, on one date.
- Row count / date span: 9,841,378 rows spanning 2026-03-01 to 2026-03-31
  (the full month), covering 331,437 distinct pages across 55 distinct clients.
- Availability: only 3,611,061 of 9,841,378 rows (~36.7%) have GSC data available,
  and just 413,966 rows (~4.2%) have GA4 data available. This confirms GA4
  coverage is far sparser than GSC — any feature relying on GA4 columns
  (pageviews, sessions, engagement) would only be usable for a small minority of
  rows, so it can't be a core feature without careful filtering or a fallback.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data can't tell me why a page's CTR is low — only that it is, relative to
peers. There's no causal structure here (no A/B test), so any recommendation
stays directional, not proof that a specific fix would work.

GA4 coverage is far sparser than GSC: confirmed in section 3, only 4.2% of rows
(413,966 of 9,841,378) have GA4 data available, versus 36.7% for GSC. Any feature
built from GA4 columns (pageviews, sessions, engagement) would only be usable for
a small minority of pages, so it can't be a core feature without explicit
filtering or a fallback for the majority of rows that lack it.

History depth is also unbalanced across clients (per dim_clients.gsc_data_start
/ ga4_data_start) — some clients have been tracked far longer than others, so
early rows for newer clients may look sparser or different not because
performance was actually worse, but because tracking simply started later.
Comparing pages across clients with very different history depth risks drawing
the wrong conclusion from a data gap rather than a real signal.

In [14]:

#No code

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.